In [18]:
# --- Step 1: Install dependencies (if not already installed)
# ! uv pip install openai requests

In [19]:
# --- Step 2: Import the library and setup base URL ---
from openai import OpenAI

In [20]:
MODEL_NAME_1, URL_1 = "google/gemma-3-27b-it", "http://localhost:8800/v1"
MODEL_NAME_2, URL_2 = "Qwen/Qwen3-VL-8B-Thinking", "http://localhost:8801/v1"
MODEL_NAME_3, URL_3 = "Qwen/Qwen2.5-VL-7B-Instruct", "http://localhost:8802/v1"
MODEL_NAME_4, URL_4 = "Qwen/Qwen2.5-VL-3B-Instruct", "http://localhost:8803/v1"

In [21]:
# Create a client pointing to vLLM's OpenAI-compatible endpoint
client = OpenAI(
    base_url=URL_4,  # Change to URL_2, URL_3, or URL_4 to test other models
    api_key="EMPTY",  # vLLM ignores auth, but the param is required
)

In [22]:
# --- Step 4: Estimate confidence from the latest response
import math
from typing import Any, Dict, List

SPECIAL_TOKENS = {
    "<|im_start|>",
    "<|im_end|>",
    "<|assistant|>",
    "<|user|>",
}


def is_special_token(token: str) -> bool:
    """
    Return True if the token should be ignored for confidence computation.

    - Exact matches: "<|im_start|>", "<|im_end|>", "<|assistant|>", "<|user|>"
    - Pure whitespace tokens are also ignored.
    """
    if token in SPECIAL_TOKENS:
        return True
    if token.strip() == "":
        return True
    return False


def extract_token_logprobs_from_choice(choice: Any) -> List[Dict[str, float]]:
    """
    Extract a list of {"token": str, "logprob": float} for the first choice's
    generated tokens, from either object-style or dict-style OpenAI responses.

    Returns an empty list if logprobs/content are missing.
    """
    logprobs = getattr(choice, "logprobs", None)
    if logprobs is None and isinstance(choice, dict):
        logprobs = choice.get("logprobs")

    if logprobs is None:
        return []

    # vLLM OpenAI-compatible: logprobs.content is a list of token entries
    content = getattr(logprobs, "content", None)
    if content is None and isinstance(logprobs, dict):
        content = logprobs.get("content")

    if not content:
        return []

    tokens: List[Dict[str, float]] = []
    for entry in content:
        if entry is None:
            continue

        # Handle both object-style and dict-style
        token = getattr(entry, "token", None)
        if token is None and isinstance(entry, dict):
            token = entry.get("token")

        logprob = getattr(entry, "logprob", None)
        if logprob is None and isinstance(entry, dict):
            logprob = entry.get("logprob")

        if token is None or logprob is None:
            continue

        try:
            tokens.append({"token": str(token), "logprob": float(logprob)})
        except (TypeError, ValueError):
            continue

    return tokens


def compute_confidence_scores(tokens: List[Dict[str, float]]) -> Dict[str, float]:
    """
    Given a list of {"token": str, "logprob": float}, filter out special tokens
    and compute several confidence metrics:

    - mean_token_prob: mean of P(y_t) over non-special tokens
    - max_token_prob: max P(y_t) over non-special tokens
    - avg_log_prob: mean of log p(y_t) over non-special tokens
    - num_tokens: number of non-special tokens

    If there are no usable tokens, returns zeros.
    """
    filtered = [t for t in tokens if not is_special_token(t["token"])]

    if not filtered:
        return {
            "mean_token_prob": 0.0,
            "max_token_prob": 0.0,
            "avg_log_prob": 0.0,
            "num_tokens": 0,
        }

    probs: List[float] = []
    for t in filtered:
        lp = t["logprob"]
        try:
            probs.append(math.exp(lp))
        except OverflowError:
            continue

    if not probs:
        return {
            "mean_token_prob": 0.0,
            "max_token_prob": 0.0,
            "avg_log_prob": 0.0,
            "num_tokens": 0,
        }

    mean_token_prob = sum(probs) / len(probs)
    max_token_prob = max(probs)
    avg_log_prob = sum(t["logprob"] for t in filtered) / len(filtered)

    return {
        "mean_token_prob": mean_token_prob,
        "max_token_prob": max_token_prob,
        "avg_log_prob": avg_log_prob,
        "num_tokens": len(filtered),
    }


def estimate_confidence_from_choice(choice: Any):
    """
    Return a confidence score (0-1) and metadata for a single choice.

    Priority:
    1. If token logprobs are available:
       - Use mean token probability as the main confidence.
    2. Otherwise:
       - Fall back to a simple text-heuristic (your original logic).
    """
    # 1) Try to use proper token logprobs
    tokens = extract_token_logprobs_from_choice(choice)
    if tokens:
        scores = compute_confidence_scores(tokens)
        confidence = scores["mean_token_prob"]  # main scalar for routing
        return confidence, {
            "source": "logprobs",
            **scores,
        }

    # 2) Fallback: heuristic based on the text content (your original approach)
    message = getattr(choice, "message", None)
    content = ""
    if isinstance(message, dict):
        content = message.get("content", "")
    elif message is not None:
        content = getattr(message, "content", "")
    text = content or ""

    uncertainty_phrases = [
        "i think",
        "maybe",
        "possibly",
        "might be",
        "could be",
        "not sure",
        "uncertain",
        "probably",
        "perhaps",
        "i believe",
    ]
    lower_text = text.lower()
    has_uncertainty = any(phrase in lower_text for phrase in uncertainty_phrases)

    confidence = 0.7
    if has_uncertainty:
        confidence -= 0.2
    if len(text.split()) < 10 and text:
        confidence += 0.1

    confidence = max(0.0, min(1.0, confidence))
    return confidence, {
        "source": "heuristic",
        "has_uncertainty": has_uncertainty,
        "token_count": len(text.split()),
    }


In [26]:
# --- Step 3: Send a simple test prompt ---
try:
    response = client.chat.completions.create(
        model="",  # e.g., "Kimi-VL-A3B-Thinking-2506"
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is 2 + 2?"}
        ],
        temperature=0.0,
        logprobs=True,
        top_logprobs=1,
        
    )

    print("✅ Request successful!")
    print("Response:\n")
    print(response.choices[0])
    
    confidence_score, confidence_info = estimate_confidence_from_choice(response.choices[0])
    print('Estimated confidence score:', confidence_score)
    print('Confidence metadata:', confidence_info)
    logprobs = getattr(response.choices[0], 'logprobs', None)
    if logprobs is None and isinstance(response.choices[0], dict):
        logprobs = response.choices[0].get('logprobs')
    if logprobs is not None:
        token_logprobs = getattr(logprobs, 'token_logprobs', None)
        if token_logprobs is None and isinstance(logprobs, dict):
            token_logprobs = logprobs.get('token_logprobs')
        print('Logprobs available tokens:', len(token_logprobs) if token_logprobs else 0)

except Exception as e:
    print("❌ Request failed:", str(e))


✅ Request successful!
Response:

Choice(finish_reason='stop', index=0, logprobs=ChoiceLogprobs(content=[ChatCompletionTokenLogprob(token='2', bytes=[50], logprob=-0.18249811232089996, top_logprobs=[TopLogprob(token='2', bytes=[50], logprob=-0.18249811232089996)]), ChatCompletionTokenLogprob(token=' +', bytes=[32, 43], logprob=-0.020119864493608475, top_logprobs=[TopLogprob(token=' +', bytes=[32, 43], logprob=-0.020119864493608475)]), ChatCompletionTokenLogprob(token=' ', bytes=[32], logprob=-1.4185804502631072e-05, top_logprobs=[TopLogprob(token=' ', bytes=[32], logprob=-1.4185804502631072e-05)]), ChatCompletionTokenLogprob(token='2', bytes=[50], logprob=-6.985420623095706e-05, top_logprobs=[TopLogprob(token='2', bytes=[50], logprob=-6.985420623095706e-05)]), ChatCompletionTokenLogprob(token=' equals', bytes=[32, 101, 113, 117, 97, 108, 115], logprob=-1.2796268463134766, top_logprobs=[TopLogprob(token=' equals', bytes=[32, 101, 113, 117, 97, 108, 115], logprob=-1.2796268463134766)]), C